In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.5 Paged attention

A contiguous KV cache wastes memory: you reserve max-length for every sequence.
**Paged attention** (vLLM) stores the cache in fixed-size pages, like OS virtual
memory, allocating pages on demand and even sharing them across sequences with a
common prefix.

In [ ]:
page_size = 4
seq_len = 10
n_pages = (seq_len + page_size - 1) // page_size      # ceil division
used = seq_len
reserved = n_pages * page_size
print(f"seq {seq_len} -> {n_pages} pages of {page_size}")
print(f"slots used {used}, reserved {reserved}, wasted {reserved - used}")

Waste is bounded by one page per sequence instead of the full max length, and pages
from a shared prompt prefix can be reused across requests. That is how vLLM serves
many concurrent users on one GPU.

In [ ]:
assert n_pages == 3
assert reserved - used < page_size      # at most one partial page wasted